In [2]:
from datasets import load_dataset

dataset = load_dataset("wikitext", "wikitext-103-raw-v1")

In [3]:
display(dataset)

DatasetDict({
    test: Dataset({
        features: ['text'],
        num_rows: 4358
    })
    train: Dataset({
        features: ['text'],
        num_rows: 1801350
    })
    validation: Dataset({
        features: ['text'],
        num_rows: 3760
    })
})

In [4]:
import nltk
from nltk.tokenize import word_tokenize
from nltk.corpus import stopwords
from nltk.stem import PorterStemmer
from nltk.stem import WordNetLemmatizer

nltk.download('punkt_tab')
nltk.download('stopwords')
nltk.download('wordnet')

[nltk_data] Downloading package punkt_tab to
[nltk_data]     C:\Users\dbere\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!
[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\dbere\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package wordnet to
[nltk_data]     C:\Users\dbere\AppData\Roaming\nltk_data...
[nltk_data]   Package wordnet is already up-to-date!


True

In [5]:
stop_words = stopwords.words("english")

def preprocess(text):
    text = text.strip()
    if len(text) == 0:
        return None
    tokens = word_tokenize(text)
    result = [word for word in tokens if word.lower() not in stop_words]
    # result = [PorterStemmer().stem(word) for word in result]
    result = [WordNetLemmatizer().lemmatize(word) for word in result]
    return result

In [6]:
text = dataset["train"][4634]["text"]
display(text)
display(preprocess(text))

' In an April 3 , 2007 interview with the Harvard Crimson , " Dershowitz confirmed that he had sent a letter last September to DePaul faculty members lobbying against Finkelstein \'s tenure . " \n'

['April',
 '3',
 ',',
 '2007',
 'interview',
 'Harvard',
 'Crimson',
 ',',
 '``',
 'Dershowitz',
 'confirmed',
 'sent',
 'letter',
 'last',
 'September',
 'DePaul',
 'faculty',
 'member',
 'lobbying',
 'Finkelstein',
 "'s",
 'tenure',
 '.',
 '``']

In [7]:
sentences = []

for item in dataset["train"].select(range(100000)):
    tokens = preprocess(item["text"])
    if tokens and len(tokens) > 2:
        sentences.append(tokens)

display(sentences)

[['=', 'Valkyria', 'Chronicles', 'III', '='],
 ['Senjō',
  'Valkyria',
  '3',
  ':',
  'Unrecorded',
  'Chronicles',
  '(',
  'Japanese',
  ':',
  '戦場のヴァルキュリア3',
  ',',
  'lit',
  '.',
  'Valkyria',
  'Battlefield',
  '3',
  ')',
  ',',
  'commonly',
  'referred',
  'Valkyria',
  'Chronicles',
  'III',
  'outside',
  'Japan',
  ',',
  'tactical',
  'role',
  '@',
  '-',
  '@',
  'playing',
  'video',
  'game',
  'developed',
  'Sega',
  'Media.Vision',
  'PlayStation',
  'Portable',
  '.',
  'Released',
  'January',
  '2011',
  'Japan',
  ',',
  'third',
  'game',
  'Valkyria',
  'series',
  '.',
  'Employing',
  'fusion',
  'tactical',
  'real',
  '@',
  '-',
  '@',
  'time',
  'gameplay',
  'predecessor',
  ',',
  'story',
  'run',
  'parallel',
  'first',
  'game',
  'follows',
  '``',
  'Nameless',
  '``',
  ',',
  'penal',
  'military',
  'unit',
  'serving',
  'nation',
  'Gallia',
  'Second',
  'Europan',
  'War',
  'perform',
  'secret',
  'black',
  'operation',
  'pitted',
  

In [8]:
# Word2vec
from gensim.models import Word2Vec

model = Word2Vec(
    sentences=sentences,
    vector_size=100,
    window=5,
    min_count=3,
    workers=4,
    sg=1
)


In [9]:
model.train(sentences, total_examples=len(sentences), epochs = 5)

(14896547, 18952055)

In [10]:
model.wv.most_similar("king")

[('throne', 0.7809875011444092),
 ('Hairan', 0.7521547675132751),
 ('annal', 0.7457359433174133),
 ('Eadberht', 0.7448541522026062),
 ('ruler', 0.7446582317352295),
 ('Domnall', 0.7438332438468933),
 ('Raedwald', 0.7437185049057007),
 ('Donnchad', 0.7408291697502136),
 ('kingship', 0.7379422187805176),
 ('kingdom', 0.7377804517745972)]

In [11]:
model.wv.similarity("cat", "dog")


np.float32(0.67887545)

In [12]:
model.wv.most_similar(negative=["king"])

[('Branch', 0.13117624819278717),
 ('Chicago', 0.11079145222902298),
 ('handled', 0.09907419979572296),
 ('analyzed', 0.09232760220766068),
 ('Savannah', 0.09213333576917648),
 ('engineering', 0.08666069060564041),
 ('Metro', 0.08551347255706787),
 ('physic', 0.07090075314044952),
 ('Engineering', 0.06953628361225128),
 ('engineer', 0.06843693554401398)]

In [13]:
model.wv.most_similar(positive=["king", "woman"], negative=["man"])

[('monarch', 0.6080366969108582),
 ('vassal', 0.5996853113174438),
 ('Domnall', 0.594489574432373),
 ('Toirdelbach', 0.5876513719558716),
 ('necropolis', 0.5822138786315918),
 ('Vazimba', 0.5787633657455444),
 ('Qedarite', 0.5785783529281616),
 ('reign', 0.5769726037979126),
 ('Radama', 0.5763570666313171),
 ('priesthood', 0.5762882828712463)]

In [ ]:
import random
import numpy as np

words = list(model.wv.index_to_key)

secret_word = random.choice(words)
print(secret_word)

similar_words = model.wv.most_similar(secret_word, topn=5000)

ranks = {secret_word: 1}

for i, (word, score) in enumerate(similar_words, start=2):
    ranks[word] = i

print("Guess the secret word.")
print("Type 'exit' to stop.")
print("Type 'hint' to get a hint.")


attempts = 0
history = []

while True:
    guess = input("Your guess: ").lower().strip()

    if guess == "exit":
        print(f"Game stopped. Secret word was: {secret_word}")
        break

    if guess == "hint":
        hint_word = similar_words[9][0]
        print(f"Hint: one close word is '{hint_word}'")
        continue

    if guess not in model.wv:
        print("This word is not in vocabulary.")
        continue

    attempts += 1

    similarity = model.wv.similarity(secret_word, guess)

    if guess == secret_word:
        print(f"Correct! The word was '{secret_word}'")
        print(f"Attempts: {attempts}")
        break

    if guess in ranks:
        rank = ranks[guess]
    else:
        rank = ">5000"

    history.append((guess, rank, similarity))

    print(f"Word: {guess}")
    print(f"Rank: {rank}")
    print(f"Similarity: {similarity:.4f}")

    print("\nBest guesses:")
    sorted_history = sorted(
        history,
        key=lambda x: x[1] if isinstance(x[1], int) else 999999
    )

    for word, rank, sim in sorted_history[:10]:
        print(f"{word:15} rank: {rank}, similarity: {sim:.4f}")

    print("-" * 40)

exists
Contexto game started!
Guess the secret word.
Type 'exit' to stop.
Type 'hint' to get a hint.


Your guess:  be


This word is not in vocabulary.
